In [ ]:
from figure_paths import DATA, RESULTS, WORK, FIGURES, setting
from pathlib import Path
from IPython.display import display

INPUT_DIR = setting("YANGTZE_ZWD_RESULTS_DIR", WORK / "zwd")
DPI = 800
CSV_COLUMNS = {}  # 可选，按原始表头指定 columns 或按指标指定 metrics。
NC_CONFIG = {}    # 可选，依据文件元数据填写 labels、roles、directions 等。


In [ ]:
def read_csv_metrics(path, override=None):
    """Extract final regional scores, in native units. Gains are B−A for R, A−B for errors."""
    import re
    from itertools import product
    import numpy as np
    import pandas as pd

    override = override or {}
    def name(value):
        s = str(value).strip().lower().replace("ω", "omega").replace("−", "-").replace("%", " percent ")
        s = re.sub(r"(?<![a-z0-9])(a|b|baseline|augmented)\s*-\s*(a|b|baseline|augmented)(?![a-z0-9])",
                   r"\1_minus_\2", s)
        return re.sub(r"[^a-z0-9]+", "_", s).strip("_")

    aliases = {
        "R": {"r", "corr", "correlation", "pearson", "pearsonr", "pcc", "pearson_r",
              "pearson_correlation", "pearson_correlation_coefficient"},
        "RMSPE": {"rmspe", "range_normalized_rmse", "range_normalised_rmse",
                  "range_normalized_root_mean_square_error", "range_normalised_root_mean_square_error"},
        "RMSE": {"rmse", "root_mean_square_error", "root_mean_squared_error"}
    }
    def metric_of(text):
        n = "_"+name(text)+"_"
        if re.search(r"_(r2|r_squared|r_square|rsquared|coefficient_of_determination|spearman)_", n):
            return "other"
        for metric in ("RMSPE", "RMSE", "R"):
            if any("_"+s+"_" in n for s in aliases[metric]):
                return metric
        if re.search(r"_(mae|mse|bias|mape|r2|ets|nrmse)_",n):
            return "other"
        return ""

    def stage_of(text):
        n = name(text)
        if re.search(r"final|residual_corrected|two_stage|baseline_plus_residual", n):
            return "final"
        if re.search(r"residual|stage_?1|first_stage|direct|uncorrected", n):
            return "other"
        return ""

    def arm_of(text):
        n = "_"+name(text)+"_"
        if re.search(r"_(a|arm_?a|group_?a|baseline|base|control|no_zwd|without_zwd)_", n):
            return "base"
        if re.search(r"_(b|arm_?b|group_?b|augmented|aug|zwd|with_zwd)_", n):
            return "zwd"
        return ""

    def sense(text):
        n = "_"+name(text)+"_"
        a = r"(?:a|arm_a|baseline|base|control|without_zwd|no_zwd)"
        b = r"(?:b|arm_b|augmented|aug|with_zwd|zwd)"
        if re.search("_"+a+"_(?:minus|less)_"+b+"_",n): return -1
        if re.search("_"+b+"_(?:minus|less)_"+a+"_",n): return 1
        return None

    for encoding in ("utf-8-sig", "gb18030"):
        try:
            df = pd.read_csv(path, encoding=encoding)
            if len(df.columns) == 1:
                df = pd.read_csv(path, sep=None, engine="python", encoding=encoding)
            break
        except UnicodeDecodeError:
            if encoding == "gb18030": raise
    names = {c:name(c) for c in df}
    meta = {
        "lead":"lead lead_time lead_h lead_hr lead_hour lead_hours lead_time_h lead_time_hour lead_time_hours forecast_hour forecast_lead forecast_lead_h forecast_lead_hour forecast_lead_hours horizon",
        "component":"component wind_component variable wind_variable field_variable output_component target_variable y_variable",
        "level":"level level_hpa pressure pressure_hpa pressure_level pressure_level_hpa isobaricinhpa plev",
        "channel":"channel channel_name channel_label output_name output_channel output_label target target_name output feature",
        "metric":"metric metric_name score_name statistic_name measure",
        "stage":"stage model_stage prediction_type prediction_stage forecast_type forecast_stage forecast_field output_type result_type product prediction prediction_field field_name field",
        "arm":"arm experiment comparison_arm model_arm system model configuration method",
    }
    common = override.get("columns", {})
    def cols(role):
        if role in common:
            if common[role] not in names: raise ValueError(f"CSV has no column {common[role]!r}")
            return [common[role]]
        return [c for c in names if names[c] in meta[role].split()]
    def one(role):
        found=cols(role)
        if len(found)>1 and any(not df[c].map(name).equals(df[found[0]].map(name)) for c in found[1:]):
            raise ValueError(f"Ambiguous CSV dimension {role}: {found}")
        return found[0] if found else None
    for c in cols("stage"):
        tags=df[c].map(stage_of)
        if tags.eq("final").any(): df=df.loc[tags.eq("final")].copy()
        elif tags.eq("other").any(): raise ValueError(f"No final-forecast records in {c!r}")
    leadcol, metriccol = one("lead"), one("metric")
    if leadcol is None: raise ValueError(f"Lead column not found: {list(names)}")
    def lead(value):
        m=re.fullmatch(r"(?:lead_?|h)?([1-6])(?:\.0+)?(?:\s*(?:h|hr|hrs|hour|hours))?",str(value).strip().lower())
        if m: return int(m[1])
    def level(value):
        m=re.search(r"(?<!\d)(85000|92500|100000|850|925|1000)(?!\d)", str(value))
        if m:
            p=int(m[1]);return p//100 if p>1000 else p
    def component(value):
        tokens=set(re.sub(r"850|925|1000","_",name(value)).split("_"))
        for v,k in (("u",{"u","zonal"}),("v",{"v","meridional"}),("omega",{"omega","wap","w","vertical"})):
            if tokens & k: return v
    def codes(columns, parser):
        candidates=[df[c].map(parser) for c in dict.fromkeys(columns)]
        valid=[v for v in candidates if v.notna().all()]
        if not valid or any(not np.array_equal(v.values, valid[0].values) for v in valid[1:]):
            raise ValueError(f"Cannot determine {parser.__name__} from {columns}")
        return valid[0]
    df["_h"]=codes([leadcol],lead)
    df["_p"]=codes(cols("level")+cols("channel")+cols("component"),level)
    df["_v"]=codes(cols("component")+cols("channel"),component)
    axes=["_h","_v","_p"]
    armcol=next((c for c in cols("arm") if df[c].map(arm_of).isin(["base","zwd"]).any()),None)
    protected={c for role in meta for c in cols(role)}
    value_cols=[c for c in names if c not in protected]
    expect=set(product(range(1,7),("u","v","omega"),(850,925,1000)))
    result={}
    for metric, gain_sign in (("R",1),("RMSPE",-1),("RMSE",-1)):
        spec=override.get("metrics",{}).get(metric,{})
        if metriccol:
            part=df.loc[df[metriccol].map(metric_of).eq(metric)]
        else:
            part=df
        if part.empty: continue
        columns=[c for c in value_cols if stage_of(names[c])!="other" and
                 (metric_of(c)==metric or (metriccol and not metric_of(c)))]
        fields={}
        def role(c):
            n=names[c]; t=set(n.split("_"))
            if t & {"pct","percent","percentage","relative","ratio","abs","absolute","valid","size","count","units","flag","class","pvalue"}:
                return None
            bound=None
            if t & {"lo","low","lower","lwr","lb","low95","lower95","p025","q025","ci025"} or re.search(r"(?:^|_)(?:2_5|0_025|025)(?:$|_)",n): bound="lo"
            if t & {"hi","high","upper","upr","ub","high95","upper95","p975","q975","ci975"} or re.search(r"(?:^|_)(?:97_5|0_975|975)(?:$|_)",n): bound="hi"
            diff=bool(t & {"gain","delta","diff","difference","improvement","improve","change","reduction"}) or sense(c) is not None
            if bound:
                if arm_of(c) and not (diff or "paired" in t): return None
                if armcol and not (diff or "paired" in t): return None
                return bound
            if t & {"bootstrap","ci","interval","confidence","stderr","se","sd","std"}: return None
            if diff: return "gain"
            arm=arm_of(c)
            if arm: return arm
            if armcol and (metric_of(c)==metric or n in {"mean","value","score","spatial_mean","spatial_means","mean_value"}):
                return "value"
        for key in ("base","zwd","gain","lo","hi","value"):
            if key in spec:
                if spec[key] not in names: raise ValueError(f"Unknown CSV column {spec[key]!r}")
                fields[key]=spec[key];continue
            candidates=[c for c in columns if role(c)==key]
            if not candidates:continue
            def rank(c):
                return (stage_of(c)=="final", metric_of(c)==metric, sense(c) is not None, "paired" in names[c])
            best=[c for c in candidates if rank(c)==max(map(rank,candidates))]
            if len(best)>1: raise ValueError(f"Ambiguous {metric}/{key} columns: {best}")
            fields[key]=best[0]
        if not fields: continue
        out=part[axes].drop_duplicates().set_index(axes)
        if set(out.index)!=expect: raise ValueError(f"{metric} lacks unique support for 6 leads, 3 components and 3 levels")
        def values(rows,c):
            a=rows[axes+[c]].copy()
            a[c]=pd.to_numeric(a[c],errors="raise");a=a.dropna(subset=[c])
            if a.groupby(axes)[c].nunique().gt(1).any(): raise ValueError(f"Multiple {metric} scores in {c} at the same coordinates")
            return a.drop_duplicates(axes).set_index(axes)[c]
        for key, col in fields.items():
            if key=="value":
                for arm in ("base","zwd"): out[arm]=values(part.loc[part[armcol].map(arm_of)==arm],col)
            elif key in ("base","zwd"): out[key]=values(part,col)
            else: out["raw_"+key]=values(part,col)
        forced=sense(spec.get("direction",""))
        if "direction" in spec and forced is None: raise ValueError("CSV direction must be A-B or B-A")
        named_direction=sense(fields.get("gain",""))
        if forced and named_direction and forced!=named_direction:
            raise ValueError(f"Explicit CSV gain direction conflicts with column header for {metric}")
        raw_direction=forced or named_direction
        if {"base","zwd"}.issubset(out):
            diff=out.zwd-out.base
            out["gain"]=gain_sign*diff
            if "raw_gain" in out:
                valid=np.isfinite(out.raw_gain) & np.isfinite(diff)
                matches=[s for s in (-1,1) if valid.any() and np.allclose(out.raw_gain[valid],s*diff[valid],rtol=1e-5,atol=1.6e-6)]
                if raw_direction and matches and raw_direction not in matches: raise ValueError("CSV difference header conflicts with A/B scores")
                if not raw_direction and len(matches)==1: raw_direction=matches[0]
        elif "raw_gain" in out and raw_direction:
            out["gain"]=gain_sign*raw_direction*out.raw_gain
        bound_senses={s for k in ("lo","hi") if (s:=sense(fields.get(k,""))) is not None}
        if len(bound_senses)>1: raise ValueError(f"Inconsistent interval orientations for {metric}")
        if forced and bound_senses and forced not in bound_senses:
            raise ValueError(f"Explicit CSV interval direction conflicts with bound headers for {metric}")
        interval_direction=next(iter(bound_senses),forced)
        if interval_direction is None and "raw_gain" in out:
            interval_direction=raw_direction
        if {"raw_lo","raw_hi"}.issubset(out):
            if (out.raw_lo>out.raw_hi).any(): raise ValueError(f"Invalid bounds for {metric}")
            if interval_direction:
                if gain_sign==interval_direction: out["lo"],out["hi"]=out.raw_lo,out.raw_hi
                else: out["lo"],out["hi"]=-out.raw_hi,-out.raw_lo
        out=out.rename_axis(index={"_h":"lead","_v":"component","_p":"level"}).reset_index()
        out.attrs["source_columns"]=fields
        out.attrs["interval_source_direction"]={1:"B-A",-1:"A-B"}.get(interval_direction)
        result[metric]=out
    if not result: raise ValueError(f"No labelled regional scores in CSV. Columns: {list(names)}")
    return result


In [ ]:
import itertools
import json
import re

import numpy as np
import pandas as pd
from types import SimpleNamespace


def _nc_summary(path, names):
    """Load only coordinate labels and small regional summaries, using installed NetCDF tools."""
    try:
        from netCDF4 import Dataset
        handle = Dataset(path)
        backend = "netCDF4"
    except ImportError:
        try:
            import xarray as xr
            handle = xr.open_dataset(path, decode_times=False, decode_timedelta=False)
            backend = "xarray"
        except ImportError:
            from scipy.io import netcdf_file
            handle = netcdf_file(path, "r", mmap=False)
            backend = "scipy"

    def attributes(x):
        if backend == "netCDF4":
            return {k: x.getncattr(k) for k in x.ncattrs()}
        return x.attrs if backend == "xarray" else x._attributes

    class Summary(dict):
        @property
        def variables(self):
            return self

    with handle:
        ds = Summary()
        ds.attrs = attributes(handle)
        ds.schema = {}
        for name, var in handle.variables.items():
            dims = tuple(var.dims if backend == "xarray" else var.dimensions)
            ds.schema[name] = dict(zip(dims, var.shape))
            dtype = var.data.dtype if backend == "scipy" else np.dtype(var.dtype)
            label = (len(dims) == 1 and var.shape[0] <= 32) or (
                len(dims) == 2 and var.shape[0] <= 32 and var.shape[1] <= 96 and dtype.kind in "SU")
            if not label and (name not in names or np.prod(var.shape) > 3000000 or any(_nc_key(d) in {"lat","latitude","lon","longitude","targettime"} for d in dims)):
                continue
            raw = var.values if backend == "xarray" else var[...]
            if np.ma.isMaskedArray(raw) and np.ma.is_masked(raw):
                raw = raw.astype(float).filled(np.nan) if raw.dtype.kind in "fiu" else raw.filled("")
            data = np.array(raw)
            if data.ndim == 2 and data.dtype.kind in "SU" and data.shape[1] <= 96:
                data = np.array(["".join(_nc_text(i) for i in row).strip() for row in data])
                dims = dims[:1]
            ds[name] = SimpleNamespace(dims=dims, sizes=dict(zip(dims,data.shape)), size=data.size,
                                       attrs=attributes(var), values=data)
        return ds


def _nc_text(x):
    if isinstance(x, (bytes, np.bytes_)):
        return x.decode("utf-8").strip("\x00 ")
    return str(x).strip()


def _nc_key(x):
    return re.sub(r"[^a-z0-9]", "", _nc_text(x).lower().replace("ω", "omega"))


_NC_ROLES = {
    "lead": {"lead", "leads", "leadtime", "forecastlead", "forecasthour", "forecasttime", "horizon"},
    "level": {"pressure", "pressurelevel", "level", "levels", "plev", "isobaricinhpa"},
    "component": {"component", "windcomponent", "variable", "windvariable", "fieldvariable", "targetvariable"},
    "channel": {"channel", "channels", "outputchannel", "targetchannel", "output", "target"},
    "arm": {"arm", "group", "model", "experiment", "variant", "configuration", "system"},
    "bound": {"bound", "bounds", "ci", "cibound", "quantile", "percentile", "endpoint", "limit", "intervalbound"},
    "metric": {"metric", "metrics", "score", "statistic", "measure", "metricname", "metricindex"},
    "rep": {"bootstrap", "bootstrapreplicate", "bootstrapreplicates", "replicate", "sample", "rep", "nboot", "boot", "draw", "bootstrapiteration"},
    "stage": {"stage", "predictiontype", "field", "forecastkind", "kind", "prediction", "product", "forecaststage","modelstage","predictionstage","forecasttype","predictionkind"},
}


def _nc_role(name, size):
    key = _nc_key(name)
    for role, aliases in _NC_ROLES.items():
        if key in aliases or key.rstrip("s") in aliases:
            if role in ("channel", "component") and size in (3, 9):
                return "channel" if size == 9 else "component"
            return role
    return ""


def _nc_list(raw, size):
    if isinstance(raw, (str, bytes)):
        s = _nc_text(raw)
        try:
            value = json.loads(s)
        except (ValueError, TypeError):
            value = [v.strip(" '\"[]()") for v in re.split(r"[,;|]", s)]
            if len(value) != size:
                value = s.split()
    else:
        value = raw
    if isinstance(value, dict):
        if all(str(i) in value for i in range(size)):
            value = [value[str(i)] for i in range(size)]
        elif all(str(i) in value for i in range(1, size + 1)):
            value = [value[str(i)] for i in range(1, size + 1)]
        elif all(isinstance(i, int) for i in value.values()):
            value = [k for k, _ in sorted(value.items(), key=lambda pair: pair[1])]
        else:
            return None
    if isinstance(value, (list, tuple, np.ndarray)) and np.ndim(value) == 1 and len(value) == size:
        return [_nc_text(v) for v in value]
    return None


def _nc_axis_labels(ds, da, dim, role, cfg):
    size = da.sizes[dim]
    user = cfg.get("labels", {})
    if dim in user or role in user:
        val = _nc_list(user[dim] if dim in user else user[role], size)
        if val is None:
            raise ValueError(f"标签数与 {dim}={size} 不一致")
        return val
    sources = []
    for name, v in ds.variables.items():
        if v.dims == (dim,) and (name == dim or _nc_role(name, size) == role and role):
            sources.append([_nc_text(x) for x in v.values])
    attributes = dict(ds.attrs)
    attributes.update(da.attrs)
    if dim in ds:
        attributes.update({dim + "_" + k: v for k, v in ds[dim].attrs.items()})
    keys = {_nc_key(dim), _nc_key(role)} | _NC_ROLES.get(role, set())
    keys |= {k+"s" for k in list(keys)}
    for k in ("coordinates","axis_labels","dimension_labels","labels","dimensions"):
        value=attributes.get(k)
        if isinstance(value,(str,bytes)):
            try:
                nested=json.loads(_nc_text(value))
            except ValueError:
                continue
            if isinstance(nested,dict): attributes.update(nested)
    for key, v in attributes.items():
        key = _nc_key(key).removesuffix("json")
        if any(key in {a + suffix for suffix in ("", "s", "order", "names", "labels", "values", "meaning", "meanings", "flagmeanings", "mapping")} for a in keys if a):
            labels = _nc_list(v, size)
            if labels:
                sources.append(labels)
    for v in sources:
        if any(re.search(r"[A-Za-zω]", x) for x in v):
            return v
    return sources[0] if sources else None


def _nc_number(x):
    numbers = re.findall(r"(?<![A-Za-z0-9.])-?\d+(?:\.\d+)?", _nc_text(x))
    if len(numbers) == 1:
        return float(numbers[0])
    simple = re.fullmatch(r"(?:lead|h|p)?\s*(\d+(?:\.\d+)?)\s*(?:hpa|pa|hours?|h)?", _nc_text(x).lower())
    return float(simple[1]) if simple else None


def _nc_comp(x):
    s = _nc_key(x)
    for c, words in {"u": ("u", "zonal", "uwind", "ucomponent"), "v": ("v", "meridional", "vwind", "vcomponent"),
                     "w": ("w", "omega", "vertical", "wcomponent", "pressurevertical")}.items():
        if s == c or any(s.startswith(a) for a in words[1:]):
            return c
    return None


def _nc_lv(x):
    results = re.findall(r"(?<![0-9])(?:85000|92500|100000|850|925|1000)(?![0-9])", _nc_text(x))
    if len(results) == 1:
        i = int(results[0])
        return i // 100 if i > 1000 else i
    return None


def _nc_final(x):
    s = _nc_key(x)
    return ("final" in s or ("corrected" in s and "uncorrected" not in s) or s in {"twostage", "stage2forecast", "combined", "residualcorrected"})


def _nc_metric(x):
    v = _nc_key(x)
    if v in {"r","corr","correlation","pearson","pearsonr","pearsoncorrelation","pearsoncorrelationcoefficient","correlationcoefficient"}:
        return "R"
    if v in {"rmspe","rangenormalizedrmse","rangenormalizedrootmeansquareerror","rangenormalisedrmse"}:
        return "RMSPE"
    if v in {"rmse","rootmeansquareerror","rootmeansquarederror"}:
        return "RMSE"
    return None


def _nc_arm(x):
    v = _nc_key(x)
    if v in {"a","arma","groupa","baseline","azwdfree","baselinewithoutzwd","withoutzwd","nozwd","abaseline","control"} or v.startswith(("abaseline","withoutzwd")):
        return "A"
    if v in {"b","armb","groupb","augmented","zwd","withzwd","baugmented","bzwd"} or v.startswith(("baugmented","withzwd")):
        return "B"
    return None


def _nc_orientation(raw, metric):
    """Sign multiplier from source contrast to a positive benefit."""
    if isinstance(raw, dict):
        value = raw.get(metric, raw.get("error" if metric!="R" else "correlation", raw.get("default")))
        return _nc_orientation(value,metric) if value is not None else None
    if raw is None:
        return None
    text = _nc_text(raw).lower().strip()
    if text.startswith("{"):
        try:
            obj = json.loads(text)
            if isinstance(obj, dict):
                return _nc_orientation({k.upper() if _nc_metric(k) else k:v for k,v in obj.items()},metric)
        except ValueError:
            pass
    text = text.replace("−","-").replace("–","-").replace("_"," ").replace("minus","-")
    text = re.sub(r"(?:without\s*zwd|no\s*zwd|base(?:line)?|control)"," a ",text)
    text = re.sub(r"(?:with\s*zwd|augmented)"," b ",text)
    text = re.sub(r"\b(?:group|arm)\s*([ab])\b",r"\1",text)
    signs = set()
    for part in re.split(r"[,;|\n]",text):
        tags = [_nc_metric(w) for w in re.findall(r"[a-z0-9]+",part)]
        if any(tags) and metric not in tags:
            continue
        if metric=="R" and re.search(r"(?:error|other) metrics?|for errors?|otherwise|all others?",part):
            continue
        if re.search(r"\bb\s*-\s*a\b",part):
            signs.add(1 if metric=="R" else -1)
        if re.search(r"\ba\s*-\s*b\b",part):
            signs.add(-1 if metric=="R" else 1)
    if len(signs)==1:
        return signs.pop()
    if not signs and (text in {"gain","benefit","improvement","positive is better","positive favors zwd"} or
                      re.search(r"positive (?:values? )?(?:means?|indicates?|favo[u]?rs?) (?:an? )?(?:improvement|better|(?:the )?augmented|(?:adding )?zwd)",text)):
        return 1
    return None


def _nc_direction(ds,var,kind,metric,config):
    override = config.get("directions",{}).get(kind)
    sign = _nc_orientation(override,metric)
    if sign is not None:
        return sign
    attrs = ds[var].attrs
    for key in ("difference_definition","direction","contrast","definition","sign_convention","description","long_name","comment"):
        sign = _nc_orientation(attrs.get(key),metric)
        if sign is not None: return sign
    prefixes = [var, kind, "paired_bootstrap_difference" if kind=="boot" else kind]
    for key,val in ds.attrs.items():
        k = _nc_key(key)
        if k in {"differencedirection","contrastdirection","differenceconvention","differencedefinition","signconvention"} or any(k==_nc_key(p)+s for p in prefixes for s in ("definition","direction","convention","sign")):
            sign = _nc_orientation(val,metric)
            if sign is not None: return sign
    return None


def _nc_declared(ds,da,cfg,var,role):
    user = cfg.get("variable_" + ("metrics" if role=="metric" else "stages"),{})
    if var in user:
        return user[var]
    keys=("metric","metric_name","score") if role=="metric" else ("stage","prediction_stage","forecast_stage","prediction_type")
    for attrs in (da.attrs,ds.attrs):
        for key in keys:
            if key in attrs: return _nc_text(attrs[key])
    return None


def _nc_specs(ds, da, cfg, metric, with_csv, var):
    """Resolve physical labels. Unlabelled choices need full-vector CSV corroboration."""
    roles = {d:cfg.get("roles",{}).get(d,_nc_role(d,n)) for d,n in da.sizes.items()}
    if any(n>24 and roles[d]!="rep" for d,n in da.sizes.items()):
        return
    labels = {d:_nc_axis_labels(ds,da,d,r,cfg) for d,r in roles.items()}
    def axes(role): return [d for d,r in roles.items() if r==role]
    leads=axes("lead"); packed=axes("channel"); levels=axes("level"); comps=axes("component")
    if len(leads)!=1: return
    l=leads[0]
    hours=[_nc_number(i) for i in labels[l] or []]
    if set(hours)!=set(range(1,7)): return
    pressures=[850,925,1000]; winds=["u","v","w"]
    coord=[]
    if len(packed)==1 and not levels and not comps:
        c=packed[0]; names=labels[c] or []
        pairs=[(_nc_comp(re.sub(r"\d+|hpa|pa","",s.lower()).strip("_- ")), _nc_lv(s)) for s in names]
        if set(pairs)!=set(itertools.product(winds,pressures)):
            cc=next((v.values for n,v in ds.items() if v.dims==(c,) and _nc_key(n) in _NC_ROLES["component"]),[])
            pp=next((v.values for n,v in ds.items() if v.dims==(c,) and _nc_key(n) in _NC_ROLES["level"]),[])
            pairs=list(zip(map(_nc_comp,cc),map(_nc_lv,pp)))
        if set(pairs)==set(itertools.product(winds,pressures)):
            coord=[((l,c),[(int(h),w,p) for h in hours for w,p in pairs],True)]
        elif with_csv and da.sizes[c]==9:
            for ps in [pressures,pressures[::-1]]:
                for cp in (list(itertools.product(winds,ps)),[(w,p) for p,w in itertools.product(ps,winds)]):
                    coord.append(((l,c),[(int(h),w,p) for h in hours for w,p in cp],False))
    elif len(levels)==1 and len(comps)==1 and not packed:
        p,c=levels[0],comps[0]; ps=[_nc_lv(x) for x in labels[p] or []]; ws=[_nc_comp(x) for x in labels[c] or []]
        if set(ps)==set(pressures) and set(ws)==set(winds):
            coord=[((l,c,p),list(itertools.product(map(int,hours),ws,ps)),True)]
        elif with_csv:
            for pp in ([ps] if set(ps)==set(pressures) else [pressures,pressures[::-1]]):
                for cc in ([ws] if set(ws)==set(winds) else [winds]):
                    coord.append(((l,c,p),list(itertools.product(map(int,hours),cc,pp)),False))
    for dims,rows,explicit in coord:
        extra=[d for d in da.dims if d not in dims and roles[d] not in ("bound","arm","rep")]
        choice=[]
        proven=explicit
        for d in extra:
            sz,lab=da.sizes[d],labels[d] or []
            index=cfg.get("isel",{}).get(d)
            if isinstance(index,dict): index=index.get(metric)
            elif roles[d]=="metric" and index is not None and list(cfg.get("metrics",()))!=[metric]:
                raise ValueError(f"NC_CONFIG.isel[{d!r}] 请选择各指标的索引，例如按元数据填写 {{'R': i, 'RMSPE': j}}。")
            if index is not None: choice.append([int(index)]);continue
            if roles[d] in ("metric","stage") and any(re.search("[A-Za-z]",s) for s in lab):
                inds=[i for i,s in enumerate(lab) if (_nc_metric(s)==metric if roles[d]=="metric" else _nc_final(s))]
                if len(inds)!=1: break
                choice.append(inds)
            elif sz==1 and roles[d] not in ("metric","stage"): choice.append([0])
            elif with_csv and sz<=12: choice.append(range(sz));proven=False
            else: break
        else:
            stage_tag=_nc_declared(ds,da,cfg,var,"stage"); metric_tag=_nc_declared(ds,da,cfg,var,"metric")
            if not axes("stage") and stage_tag is not None and not _nc_final(stage_tag): continue
            if not axes("metric") and metric_tag is not None and _nc_metric(metric_tag)!=metric: continue
            stageok=bool(axes("stage")) or _nc_final(stage_tag or "") or cfg.get("final_only") is True
            metricok=bool(axes("metric")) or _nc_metric(metric_tag or "")==metric
            if not (stageok and metricok) and not with_csv:continue
            if np.prod([len(k) for k in choice])>192: continue
            for idx in itertools.product(*choice):
                yield dims, rows, dict(zip(extra,idx)), roles, labels, (proven and stageok and metricok)


def _nc_extract(ds, var, mode, metric, cfg, with_csv):
    if var not in ds: return
    da=ds[var]
    side={"ci":"bound","boot":"rep","arms":"arm"}.get(mode)
    for dims,rows,sel,roles,labels,proven in _nc_specs(ds,da,cfg,metric,with_csv,var):
        remain=[d for d in da.dims if d not in sel]; rest=[d for d in remain if d not in dims]
        if (side and (len(rest)!=1 or roles[rest[0]]!=side)) or (not side and rest): continue
        arr=da.values[tuple(sel.get(d,slice(None)) for d in da.dims)]
        arr=np.transpose(arr,[remain.index(d) for d in (*dims,*rest)]).astype(float).reshape(len(rows),-1)
        keys=pd.MultiIndex.from_tuples([(h,"omega" if w=="w" else w,p) for h,w,p in rows],names=["lead","component","level"])
        if len(keys)!=54 or not keys.is_unique:continue
        layout=(tuple(dims),tuple(rows))
        if mode=="arms":
            values=[_nc_arm(s) for s in labels[rest[0]] or []]
            orders=[(values.index("A"),values.index("B"))] if values.count("A")==values.count("B")==1 else []
            if not orders and with_csv and arr.shape[1]==2: orders=[(0,1),(1,0)];proven=False
            for a,b in orders:
                df=pd.DataFrame({"base":arr[:,a],"zwd":arr[:,b]},index=keys)
                df["gain"]=(df.zwd-df.base)*(1 if metric=="R" else -1)
                yield df.sort_index(), layout,sel,proven
        elif mode=="gain":
            yield pd.DataFrame({"value":arr[:,0]},index=keys).sort_index(),layout,sel,proven
        elif mode=="ci":
            if arr.shape[1]!=2: continue
            a,b=arr[:,0],arr[:,1]; finite=np.isfinite(a)&np.isfinite(b)
            if np.any(a[finite]>b[finite]) and np.any(a[finite]<b[finite]): continue
            if np.any(a[finite]>b[finite]):a,b=b,a
            yield pd.DataFrame({"lo":a,"hi":b},index=keys).sort_index(),layout,sel,proven
        elif mode=="boot" and arr.shape[1]==1000:
            bounds=np.quantile(arr,[.025,.975],axis=1)
            yield pd.DataFrame({"lo":bounds[0],"hi":bounds[1]},index=keys).sort_index(),layout,sel,proven


def _nc_same(a,b):
    return np.allclose(a,b, rtol=5e-6, atol=1.2e-7, equal_nan=True)


def _nc_consistent(frame,ref):
    if ref is None or "gain" not in ref or not np.isfinite(ref.gain).all(): return False
    if not frame.index.equals(ref.index) or not _nc_same(frame.gain,ref.gain): return False
    return all(_nc_same(frame[k],ref[k]) for k in ("base","zwd") if k in frame and k in ref)


def _nc_compatible(layout, s1, layout2, s2):
    return (layout[0]==layout2[0] and layout[1]==layout2[1]) and all(s1[k]==s2[k] for k in s1.keys()&s2.keys())


def _nc_identity_proven(ds,var,sel,metric,cfg,matched_points):
    """An interval needs its own metric/stage identity or the identical shared index."""
    da=ds[var]
    roles={d:cfg.get("roles",{}).get(d,_nc_role(d,s)) for d,s in da.sizes.items()}
    for role in ("metric","stage"):
        dims=[d for d,r in roles.items() if r==role]
        if not dims:
            declared=_nc_declared(ds,da,cfg,var,role)
            if role=="metric" and _nc_metric(declared)!=metric:return False
            if role=="stage" and not (_nc_final(declared or "") or cfg.get("final_only") is True):return False
            continue
        if len(dims)!=1:return False
        d=dims[0]; i=sel.get(d)
        if i is None: return False
        labels=_nc_axis_labels(ds,da,d,role,cfg)
        if labels:
            label=labels[i]
            if role=="metric" and _nc_metric(label)==metric:continue
            if role=="stage" and _nc_final(label):continue
        override=cfg.get("isel",{}).get(d)
        if role=="stage" and override is not None: continue
        if role=="metric" and isinstance(override,dict) and metric in override: continue
        if not matched_points or not any(p[2].get(d)==i for p in matched_points): return False
    return True


def read_nc_metrics(path, csv_tables=None, settings=None):
    """Return labelled final-forecast spatial mean gains and any verifiable paired intervals."""
    cfg=settings or {}; source=csv_tables or {}
    vr={"arms":"spatial_means","gain":"improvement","ci":"confidence_interval",
        "boot":"paired_bootstrap_difference",**cfg.get("variables",{})}
    ds=_nc_summary(path,set(vr.values()))
    out={}; names=["lead","component","level"]
    for metric in cfg.get("metrics",("R","RMSPE","RMSE")):
        ref=source.get(metric)
        if ref is not None:
            ref=ref.set_index(names).sort_index()
            if len(ref)!=54 or not ref.index.is_unique: ref=None
        hasref=ref is not None and "gain" in ref and np.isfinite(ref.gain).all()
        points=[]
        for mode in ("arms","gain"):
            for df,layout,sel,proven in _nc_extract(ds,vr[mode],mode,metric,cfg,hasref):
                if mode=="gain":
                    sg=_nc_direction(ds,vr[mode],"gain",metric,cfg)
                    signs=[sg] if sg else [1,-1] if hasref else []
                else:signs=[1]
                for s in signs:
                    pt=df.copy()
                    if mode=="gain":pt["gain"]=s*pt.value
                    if not np.isfinite(pt.gain).all():continue
                    if hasref and not _nc_consistent(pt,ref):continue
                    if proven or hasref:
                        pt.attrs["nc_variable"]=vr[mode]
                        points.append((pt,layout,sel))
        arm_points=[p for p in points if {"base","zwd"}.issubset(p[0])]
        if arm_points:
            points=arm_points
        if not points and not hasref:continue
        if hasref:
            result=ref.copy()
            if points:
                for key in ("gain","base","zwd"):
                    if key in points[0][0]:
                        result[key]=points[0][0][key]
        else:
            result=points[0][0].copy()
            if any(not _nc_same(result.gain,p.gain) for p,_,_ in points):
                raise ValueError(f"{metric}: ambiguous or inconsistent labelled NC effect estimates; check NC_CONFIG and file metadata.")
        result["lo"]=np.nan
        result["hi"]=np.nan
        original=ref if ref is not None and all(k in ref for k in ["lo","hi"]) else None
        sources=[]
        for mode in ("ci","boot"):
            if result[["lo","hi"]].notna().all().all():break
            options=[]
            for ci,layout,sel,proven in _nc_extract(ds,vr[mode],mode,metric,cfg,hasref):
                related=[p for p in points if _nc_compatible(layout,sel,p[1],p[2])]
                if not (proven or related):continue
                sg=_nc_direction(ds,vr[mode],mode,metric,cfg)
                rel=_nc_text(ds[vr[mode]].attrs.get("of", ds[vr[mode]].attrs.get("source_variable",""))).lower()
                if sg is None and rel==vr["gain"].lower() and vr["gain"] in ds: sg=_nc_direction(ds,vr["gain"],"gain",metric,cfg)
                for sign in ([sg] if sg else [1,-1] if original is not None else []):
                    aligned=ci if sign==1 else pd.DataFrame({"lo":-ci.hi,"hi":-ci.lo},index=ci.index)
                    full_ci_match=False
                    if original is not None:
                        use=original[["lo","hi"]].notna().all(axis=1)&aligned.notna().all(axis=1)
                        if use.any() and not _nc_same(original.loc[use,["lo","hi"]],aligned.loc[use]):continue
                        full_ci_match=bool(use.all())
                        if sg is None and not full_ci_match:continue
                    if full_ci_match or _nc_identity_proven(ds,vr[mode],sel,metric,cfg,related):
                        options.append(aligned)
            if options and all(_nc_same(ci.values,options[0].values) for ci in options):
                miss=result[["lo","hi"]].isna().any(axis=1)
                result.loc[miss,["lo","hi"]]=options[0].loc[miss,["lo","hi"]]
                sources.append(vr[mode])
        results=result[[k for k in ("base","zwd","gain","lo","hi") if k in result]].reset_index()
        results.attrs["point_source"]=(points[0][0].attrs["nc_variable"]+(" (corroborated by CSV)" if hasref else "") if points
                                      else "CSV; no independent NC point estimate")
        results.attrs["ci_source"]=", ".join(sources) or "no labelled paired 95% CI with verified orientation"
        results.attrs["nc_schema"]=ds.schema
        out[metric]=results
    return out


In [ ]:
import numpy as np
import pandas as pd
from itertools import product
from matplotlib.ticker import MaxNLocator

KEY = ["lead", "component", "level"]
INDEX = pd.MultiIndex.from_tuples(list(product(range(1,7), ("u","v","omega"),
                                               (850,925,1000))), names=KEY)
METRICS = {"R": "FigS22", "RMSPE": "FigS23"}
csv_path = INPUT_DIR / "zwd_matched_spatial_summary.csv"
nc_path = INPUT_DIR / "zwd_matched_comparison_metrics.nc"
if not csv_path.is_file() and not nc_path.is_file():
    raise FileNotFoundError(f"请检查第 1 个代码框的 INPUT_DIR。{INPUT_DIR} 中需要空间汇总 CSV 或正式指标 NC。")

def indexed(frame):
    view = frame.set_index(KEY)
    if not view.index.is_unique or set(view.index) != set(INDEX):
        raise ValueError("每项指标需覆盖 54 个唯一的时效、分量和气压层组合。")
    return view.reindex(INDEX)

def combine_metric(metric, csv_frame, nc_frame):
    frames = [(where, indexed(df)) for where,df in (("NC", nc_frame),("CSV", csv_frame)) if df is not None]
    out = pd.DataFrame(index=INDEX, columns=["gain","lo","hi"], dtype=float)
    out["point_source"] = ""
    out["ci_source"] = ""
    for where, df in frames:
        if "gain" in df:
            val = pd.to_numeric(df.gain)
            shared = out.gain.notna() & val.notna()
            if not np.allclose(out.loc[shared,"gain"], val.loc[shared], rtol=2e-5, atol=2e-6):
                raise ValueError(f"{metric} 的 NC 与 CSV 点估计不一致，请核对来源和精度。")
            fill = out.gain.isna() & np.isfinite(val)
            out.loc[fill, "gain"] = val[fill]
            out.loc[fill, "point_source"] = df.attrs.get("point_source", where)
        if {"lo","hi"}.issubset(df):
            lo, hi = pd.to_numeric(df.lo), pd.to_numeric(df.hi)
            have = np.isfinite(lo) & np.isfinite(hi)
            if (lo[have] > hi[have]).any():
                raise ValueError(f"{metric} 的 {where} 置信区间下界大于上界。")
            both = have & out.lo.notna() & out.hi.notna()
            if not (np.allclose(out.lo[both],lo[both],atol=2e-6,rtol=2e-5)
                    and np.allclose(out.hi[both],hi[both],atol=2e-6,rtol=2e-5)):
                raise ValueError(f"{metric} 的 NC 与 CSV 的配对区间不一致，请核实。")
            fill = have & out.lo.isna()
            out.loc[fill,"lo"], out.loc[fill,"hi"] = lo[fill], hi[fill]
            out.loc[fill,"ci_source"] = where
    if not np.isfinite(out.gain).all():
        if metric=="RMSE":
            return None
        raise ValueError(f"未能唯一读取 {metric} 的 54 个最终预报点估计。请查看上方字段诊断。")
    if metric=="R" and (out.gain.abs() > 2.000001).any():
        raise ValueError("相关系数差的有效范围为 [−2,2]。")
    return out.reset_index()

globals().pop("figures", None)
globals().pop("zwd_tables", None)
csv_tables, nc_tables = {}, {}
if csv_path.exists():
    try:
        csv_spec = CSV_COLUMNS if not CSV_COLUMNS or "metrics" in CSV_COLUMNS or "columns" in CSV_COLUMNS else {"columns":CSV_COLUMNS}
        csv_tables = read_csv_metrics(csv_path, csv_spec)
    except (ValueError, OSError, KeyError) as exc:
        print(f"CSV 结构提示：{exc}")
if nc_path.exists():
    try:
        nc_tables = read_nc_metrics(nc_path, csv_tables=csv_tables, settings=NC_CONFIG)
    except (ValueError, OSError, KeyError, ImportError) as exc:
        print(f"NC 结构提示：{exc}")
for name, table in nc_tables.items():
    print(f"{name} NC 配对区间来源: {table.attrs.get('ci_source', '—')}")
for name, table in csv_tables.items():
    print(f"{name} CSV: {table.attrs.get('source_columns', {})}")
zwd_tables, metric_issues = {}, []
for metric in METRICS:
    try:
        table = combine_metric(metric, csv_tables.get(metric), nc_tables.get(metric))
    except ValueError as exc:
        if metric != "RMSE":
            metric_issues.append(str(exc))
        print(f"{metric}: {exc}")
        continue
    if table is not None:
        zwd_tables[metric] = table
        count = int(table.lo.notna().sum())
        print(f"{metric}: 54 个点估计，{count}/54 个配对区间。")
        if count<54:
            print("  无法核实区间的组合标 ×；若文件有区间而方向未明，请按正式元数据设置第 1 框。")

if metric_issues:
    raise ValueError("\n".join(metric_issues))


In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator


def panel_xaxis(part):
    """根据本子图的数据确定对称轴限、五个主刻度及科学计数倍率。"""
    values = part[["gain", "lo", "hi"]].to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError("当前子图没有可绘制的点估计或区间端点。")

    peak = float(np.max(np.abs(values)))
    limit = 1.07 * peak if peak > 0 else 0.001

    target_step = limit / 2
    unit = 10.0 ** np.floor(np.log10(target_step))
    nice_steps = unit * np.array(
        [1, 1.5, 2, 2.5, 3, 4, 5, 6, 7.5, 8, 10]
    )
    step = nice_steps[nice_steps <= target_step * (1 + 1e-12)][-1]
    ticks = step * np.arange(-2, 3)

    exponent = int(np.floor(np.log10(2 * step)))
    scale = 10.0 ** exponent
    labels = [
        "0" if t == 0 else f"{t / scale:g}".replace("-", "−")
        for t in ticks
    ]
    return limit, ticks, labels, exponent


def make_interval_figure(data, metric):
    levels = (850, 925, 1000)
    components = ("u", "v", "omega")
    titles = (
        r"Zonal wind, $u$",
        r"Meridional wind, $v$",
        r"Pressure vertical velocity, $\omega$",
    )
    colors = {
        "better": "#10675e",
        "worse": "#ad4a2a",
        "cross": "#606d7b",
        "absent": "#46484b",
    }
    xlabel = (
        r"$\Delta R$ (with − without ZWD)"
        if metric == "R"
        else r"$\Delta$RMSPE (without − with ZWD)"
    )

    fig, axes = plt.subplots(
        3, 3, figsize=(10.0, 7.9), dpi=112,
        sharex=False, sharey=True,
    )
    fig.subplots_adjust(
        left=0.105, right=0.975, bottom=0.148, top=0.91,
        wspace=0.26, hspace=0.48,
    )

    used = set()

    for row, level in enumerate(levels):
        for col, component in enumerate(components):
            ax = axes[row, col]
            part = data.loc[
                data["level"].eq(level) & data["component"].eq(component)
            ].sort_values("lead")

            limit, ticks, labels, exponent = panel_xaxis(part)

            ax.set_xlim(-limit, limit)
            ax.xaxis.set_major_locator(FixedLocator(ticks))
            ax.xaxis.set_major_formatter(FixedFormatter(labels))
            ax.xaxis.set_minor_locator(NullLocator())

            ax.set_yticks(range(1, 7))
            ax.set_ylim(6.52, 0.48)
            ax.tick_params(
                axis="both", length=3, width=0.7, pad=3, labelsize=10.4
            )
            ax.tick_params(axis="x", labelbottom=True)
            ax.tick_params(
                axis="y", left=(col == 0), labelleft=(col == 0)
            )

            ax.axvline(
                0, color="#73797c", lw=0.9,
                ls=(0, (3, 2)), zorder=1,
            )

            letter = chr(ord("a") + 3 * row + col)
            ax.text(
                0.005, 1.015, f"({letter})",
                transform=ax.transAxes,
                ha="left", va="bottom", fontsize=10.5,
            )
            ax.text(
                0.995, 1.015, rf"$\times 10^{{{exponent}}}$",
                transform=ax.transAxes,
                ha="right", va="bottom", fontsize=10.5,
            )

            for pt in part.itertuples():
                if np.isfinite(pt.lo) and np.isfinite(pt.hi):
                    if pt.lo > 0:
                        group = "better"
                    elif pt.hi < 0:
                        group = "worse"
                    else:
                        group = "cross"

                    color = colors[group]

                    y_ci = pt.lead + 0.14
                    ax.hlines(
                        y_ci, pt.lo, pt.hi,
                        color=color, lw=1.45, zorder=2,
                    )
                    ax.vlines(
                        np.unique([pt.lo, pt.hi]),
                        y_ci - 0.08, y_ci + 0.08,
                        color=color, lw=1.05, zorder=2,
                    )
                    ax.plot(
                        pt.gain, pt.lead - 0.13, "o",
                        ms=4.2, mec=color, mew=1,
                        mfc="white" if group == "cross" else color,
                        zorder=3,
                    )
                else:
                    group = "absent"
                    ax.plot(
                        pt.gain, pt.lead, "x",
                        color=colors[group], ms=5, mew=1.1,
                    )

                used.add(group)

            for edge in ("top", "right"):
                ax.spines[edge].set_visible(False)
            for edge in ("bottom", "left"):
                ax.spines[edge].set_color("#a8afb4")
                ax.spines[edge].set_linewidth(0.65)

            if row == 0:
                ax.set_title(titles[col], fontsize=12, pad=18)
            if col == 0:
                ax.set_ylabel(
                    f"{level} hPa\nLead time (h)",
                    fontsize=11.5, labelpad=9,
                )

    fig.supxlabel(xlabel, x=0.54, y=0.076, fontsize=12)

    legend_items = (
        ("better", "CI above zero"),
        ("cross", "CI includes zero"),
        ("worse", "CI below zero"),
        ("absent", "CI unavailable"),
    )
    handles = [
        Line2D(
            [], [], linestyle="none",
            marker="x" if key == "absent" else "o",
            color=colors[key],
            markerfacecolor="white" if key == "cross" else colors[key],
            markersize=5, label=label,
        )
        for key, label in legend_items if key in used
    ]
    fig.legend(
        handles=handles, loc="lower center",
        bbox_to_anchor=(0.54, 0.011),
        ncol=len(handles), frameon=False,
        fontsize=10.5, handlelength=1.1,
    )
    return fig


with mpl.rc_context({"font.family": "DejaVu Sans", "font.size": 11}):
    figures = {
        metric: make_interval_figure(zwd_tables[metric], metric)
        for metric in ("R", "RMSPE")
    }
    plt.show()


In [ ]:
from PIL import Image
for metric, stem in METRICS.items():
    figures[metric].savefig(FIGURES / f'{stem}.png', dpi=DPI, facecolor='white')
    with Image.open(FIGURES / f'{stem}.png') as image:
        image.convert('RGB').save(FIGURES / f'{stem}.tiff', compression='tiff_lzw', dpi=(DPI, DPI))
